# Network-Size Residual Explorer

This notebook reloads the saved adversarial residual study from CPU-friendly activation caches. It does not require GPU inference once the `.pt` cache files exist.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

In [ ]:
ROOT = Path.cwd()
CACHE_DIR = ROOT / 'cache'
ACT_DIR = CACHE_DIR / 'activations' / 'adversarial'
PLOTS_DIR = ROOT / 'plots'
SUMMARY_JSON = CACHE_DIR / 'network_size_residual_summary_adversarial.json'
SUMMARY_CSV = CACHE_DIR / 'network_size_residual_summary_adversarial.csv'
LEGEND_MAIN = CACHE_DIR / 'network_size_residual_legend_adversarial_main.txt'
LEGEND_SUPP = CACHE_DIR / 'network_size_residual_legend_adversarial_supplement.txt'
LEGEND_TXT = CACHE_DIR / 'network_size_residual_legend_adversarial.txt'
LATEX_TABLE = CACHE_DIR / 'network_size_residual_summary_adversarial.tex'
PAPER_LATEX_TABLE = CACHE_DIR / 'network_size_residual_summary_adversarial_paper.tex'
SUPPLEMENT_STATS_CSV = CACHE_DIR / 'network_size_residual_supplement_stats.csv'
SUPPLEMENT_STATS_TEX = CACHE_DIR / 'network_size_residual_supplement_stats.tex'
AMPLIFICATION_FIG = PLOTS_DIR / 'network_size_residual_amplification_adversarial.png'

summary = json.loads(SUMMARY_JSON.read_text(encoding='utf-8'))
table = pd.read_csv(SUMMARY_CSV)
table

In [ ]:
def load_activation_bundle(label):
    slug = label.lower().replace('/', '_').replace(' ', '_').replace('-', '_').replace('.', 'p')
    return torch.load(ACT_DIR / f'{slug}.pt', map_location='cpu', weights_only=False)

bundles = {item['label']: load_activation_bundle(item['label']) for item in summary}
list(bundles)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
regimes = ['early_id', 'mid_id', 'late_id']
regimes_ood = ['early_ood', 'mid_ood', 'late_ood']
titles = ['Early layers', 'Mid layers', 'Late layers']
for ax, id_key, ood_key, title in zip(axes, regimes, regimes_ood, titles):
    x = table['params_log10'].to_numpy()
    ax.plot(x, table[id_key].to_numpy(), marker='o', linewidth=2.0, label='Within')
    ax.plot(x, table[ood_key].to_numpy(), marker='D', linewidth=2.0, linestyle='--', label='Adversarial')
    for _, row in table.iterrows():
        ax.text(row['params_log10'] + 0.01, row[ood_key] + 0.003, row['label'], fontsize=8)
    ax.set_title(title)
    ax.set_xlabel('log10(parameters)')
    ax.set_ylabel('Mean relative residual')
    ax.grid(True, linestyle=':', alpha=0.35)
axes[0].legend(frameon=False)
plt.show()

In [ ]:
selected_model = 'Qwen 7B'
bundle = bundles[selected_model]
within_states = np.stack(bundle['sets']['Within']['last_token_states'], axis=0).astype(np.float32)
adv_states = np.stack(bundle['sets']['Adversarial']['last_token_states'], axis=0).astype(np.float32)

def pca_project(state_array, n_components=3):
    flat = state_array.reshape(-1, state_array.shape[-1])
    flat = flat - flat.mean(axis=0, keepdims=True)
    u, s, vh = np.linalg.svd(flat, full_matrices=False)
    basis = vh[:n_components]
    projected = flat @ basis.T
    explained = (s[:n_components] ** 2) / np.maximum((s ** 2).sum(), 1e-12)
    return projected.reshape(state_array.shape[0], state_array.shape[1], n_components), explained

within_pca, within_var = pca_project(within_states)
adv_pca, adv_var = pca_project(adv_states)
fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
for idx in range(min(8, within_pca.shape[0])):
    ax1.plot(within_pca[idx, :, 0], within_pca[idx, :, 1], within_pca[idx, :, 2], alpha=0.75)
for idx in range(min(8, adv_pca.shape[0])):
    ax2.plot(adv_pca[idx, :, 0], adv_pca[idx, :, 1], adv_pca[idx, :, 2], alpha=0.75)
ax1.set_title(f'{selected_model} within\nexplained var={within_var.round(3)}')
ax2.set_title(f'{selected_model} adversarial\nexplained var={adv_var.round(3)}')
plt.show()

In [ ]:
print(LEGEND_TXT.read_text(encoding='utf-8'))

In [ ]:
amplification = table[['label', 'params', 'params_log10']].copy()
amplification['early_delta'] = table['early_ood'] - table['early_id']
amplification['mid_delta'] = table['mid_ood'] - table['mid_id']
amplification['late_delta'] = table['late_ood'] - table['late_id']
amplification['mean_delta'] = amplification[['early_delta', 'mid_delta', 'late_delta']].mean(axis=1)
display(amplification.sort_values('mean_delta', ascending=False))

for column in ['early_delta', 'mid_delta', 'late_delta']:
    print(f'\nTop models by {column}:')
    display(amplification[['label', column]].sort_values(column, ascending=False).head(5))

In [ ]:
print('Main-figure legend:\n')
print(LEGEND_MAIN.read_text(encoding='utf-8'))
print('\nSupplement legend:\n')
print(LEGEND_SUPP.read_text(encoding='utf-8'))

In [ ]:
print(LATEX_TABLE.read_text(encoding='utf-8'))

In [ ]:
supplement_stats = pd.read_csv(SUPPLEMENT_STATS_CSV)
supplement_stats

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(AMPLIFICATION_FIG)))

In [ ]:
print(PAPER_LATEX_TABLE.read_text(encoding='utf-8'))
print('\n--- Supplement table ---\n')
print(SUPPLEMENT_STATS_TEX.read_text(encoding='utf-8'))